# Data Quality Audit of Raw Data from the Investing.com Data Platform

The main objective of the data quality audit is to assess the quality, consistency, and general characteristics of the raw data obtained from the Investing.com Data Platform.

The analysis focuses on the following aspects:
* Missing data and missing observations
* Index integrity and consistency
* Duplicated indices
* Timestamp and time zone consistency
* Overall data characteristics

**All data and variables fetched from Investing.com are explained in the [Investing.com data overview](./01_Investing_com_data_overview.ipynb).**

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
)
from src.utils.data_quality import (
    print_index_summary,
    print_missing_values,
    find_missing_dates,
    print_data_overview,
)
from src.utils.file_io import load_investing_data

In [2]:
data_inv = load_investing_data(
    DATA_DIR / "raw" / "investing_com",
)

## 1. Investing.com data

### 1.1 Missing Data

In [3]:
for name, data in data_inv.items():
    print_missing_values(data=data, name=name)


--- [dutch_ttf_natural_gas_futures] Missing values ---
Vol.    1

--- [eu_carbon_emissions_futures] Missing values ---
Vol.    4

--- [eur_pln] Missing values ---
Vol.    1148

--- [rotterdam_coal_futures] Missing values ---
Vol.    244


### 1.2 Missing Data – Observations

Missing values occur only in the `Vol.` column; `Price`, `Open`, `High` and `Low` are complete in all four datasets. `Vol.` is entirely empty in `eur_pln` (1148 rows), partly empty in `rotterdam_coal_futures` (244 of 1135 rows) and almost complete in `eu_carbon_emissions_futures` (4 of 1129 empty) and `dutch_ttf_natural_gas_futures` (1 of 1158).

Since the price series contain no NaN values, all gaps appear only in the time index (see 1.4).

### 1.3 Index Integrity

In [4]:
for name, data in data_inv.items():
    print_index_summary(data=data, name=name)


--- [dutch_ttf_natural_gas_futures] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  -1 days +00:00:00: 925
  -3 days +00:00:00: 195
  -2 days +00:00:00: 29
  -5 days +00:00:00: 5
  -4 days +00:00:00: 3

--- [eu_carbon_emissions_futures] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  -1 days +00:00:00: 896
  -3 days +00:00:00: 222
  -5 days +00:00:00: 6
  -2 days +00:00:00: 2
  -4 days +00:00:00: 2

--- [eur_pln] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  -1 days +00:00:00: 918
  -3 days +00:00:00: 229

--- [rotterdam_coal_futures] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  -1 days +00:00:00: 901
  -3 days +00:00:00: 221
  -4 days +00:00:00: 8
  -2 days +00:00:00: 4


In [5]:
for name, data in data_inv.items():
    missing_index = find_missing_dates(data=data, name=name, freq="D")
    print("missing indicies for Monday-Friday")
    print([i for i in missing_index if i.day_of_week < 5])


--- [dutch_ttf_natural_gas_futures] Missing timestamps (freq=D) ---
Count: 448

Missing ranges:
  2022-06-04 00:00:00+00:00 - 2022-06-05 00:00:00+00:00 (2 timestamps)
  2022-06-11 00:00:00+00:00 - 2022-06-12 00:00:00+00:00 (2 timestamps)
  2022-06-18 00:00:00+00:00 - 2022-06-19 00:00:00+00:00 (2 timestamps)
  2022-06-25 00:00:00+00:00 - 2022-06-26 00:00:00+00:00 (2 timestamps)
  2022-07-02 00:00:00+00:00 - 2022-07-03 00:00:00+00:00 (2 timestamps)
  2022-07-09 00:00:00+00:00 - 2022-07-10 00:00:00+00:00 (2 timestamps)
  2022-07-16 00:00:00+00:00 - 2022-07-17 00:00:00+00:00 (2 timestamps)
  2022-07-23 00:00:00+00:00 - 2022-07-24 00:00:00+00:00 (2 timestamps)
  2022-07-30 00:00:00+00:00 - 2022-07-31 00:00:00+00:00 (2 timestamps)
  2022-08-06 00:00:00+00:00 - 2022-08-07 00:00:00+00:00 (2 timestamps)
  2022-08-13 00:00:00+00:00 - 2022-08-14 00:00:00+00:00 (2 timestamps)
  2022-08-20 00:00:00+00:00 - 2022-08-21 00:00:00+00:00 (2 timestamps)
  2022-08-27 00:00:00+00:00 - 2022-08-28 00:00:00+0

### 1.4 Index Integrity – Observations

All datasets are tz-aware (UTC) and contain no duplicated indices. All of them start on 2022-06-01 and the index is sorted in descending order (newest first). `dutch_ttf_natural_gas_futures` and `eur_pln` end on 2026-10-23, `eu_carbon_emissions_futures` and `rotterdam_coal_futures` on 2026-10-22, so the data extend 1–2 days beyond the download end date (2026-10-21).

Time steps are dominated by 1-day steps and by 3-day steps that span weekends. Every dataset lacks 229 Saturdays, and 229 Sundays (`dutch_ttf_natural_gas_futures`: 202 Sundays, because it is also quoted on some Sundays), i.e. non-trading days. `eur_pln` has no other gaps, and its longest step is 3 days.

The remaining weekday gaps are few: 12 in `rotterdam_coal_futures`, 17 in `dutch_ttf_natural_gas_futures` and 18 in `eu_carbon_emissions_futures`. They coincide with public holidays (Christmas, 26 December, New Year, Good Friday, Easter Monday; for coal also 2 January 2023). The longest gap is 5 days (`dutch_ttf_natural_gas_futures` and `eu_carbon_emissions_futures` over Easter) and 4 days in `rotterdam_coal_futures`.

### 1.5 Data Descriptive Statistics

In [6]:
for name, data in data_inv.items():
    print_data_overview(data=data, name=name)


--- [dutch_ttf_natural_gas_futures] Overview ---
Shape: 1158 x 7

Info:
<class 'pandas.DataFrame'>
DatetimeIndex: 1158 entries, 2026-10-23 00:00:00+00:00 to 2022-06-01 00:00:00+00:00
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      1158 non-null   str    
 1   Price     1158 non-null   float64
 2   Open      1158 non-null   float64
 3   High      1158 non-null   float64
 4   Low       1158 non-null   float64
 5   Vol.      1157 non-null   str    
 6   Change %  1158 non-null   str    
dtypes: float64(4), str(3)
memory usage: 97.1 KB

Description:
             Price         Open         High          Low
count  1158.000000  1158.000000  1158.000000  1158.000000
mean     56.350521    56.447284    58.739447    54.291498
std      44.953450    45.186769    48.116313    42.336649
min      22.934000    22.860000    23.505000    22.315000
25%      32.962500    33.000000    33.796250    32.106250
50%      40.374000    

In [7]:
print("Coefficient of Variation (CV)")
for name, data in data_inv.items():
    cv = data["Price"].std() / data["Price"].mean()
    print(f"dataset: {name}, price CV = {cv.round(2)}")

Coefficient of Variation (CV)
dataset: dutch_ttf_natural_gas_futures, price CV = 0.8
dataset: eu_carbon_emissions_futures, price CV = 0.12
dataset: eur_pln, price CV = 0.04
dataset: rotterdam_coal_futures, price CV = 0.51


## 2. Conclusions and Data Cleaning / Preprocessing Decisions

**Main findings**
`Price` is complete in all datasets; NaN values are limited to `Vol.`. Indices are UTC and free of duplicates, but sorted in descending order. Index gaps are structural (weekends and public holidays in all datasets); the longest gap is 5 days.

### 2.1 Data Cleaning Decisions

**All datasets**

* Keep only `Price`; drop `Vol.` (empty or partly empty), `Open`, `High`, `Low` (redundant with `Price`), `Change %` (derived from `Price`) and `Date` (duplicates the index).
* No deduplication or timezone conversion is required.

### 2.2 Data Preprocessing Decisions

**Time Index**

* Sort the index in ascending order before any fill operation.
* Reindex all datasets to a complete UTC index (2022-06-01 – 2026-10-21), so that they can be aligned with the other sources. The rows after 2026-10-21 are dropped.

**Missing timestamps**

* Fill all missing days (weekends and holidays) with forward fill (FFILL) of `Price`. This carries over the last available market price and does not introduce artificial price movements, unlike interpolation. All gaps are short (up to 5 days) and no dataset has missing values at the start of the sample.